# Phase 3.2 — Session 1: Byzantine-Robust Aggregation Comparison

Runs **7 aggregation methods** sequentially for 40 FL rounds each.
No diffusion. No blockchain for methods A–F. Ganache only for method G.

| # | Method | Defense Mechanism | ~Time |
|:-:|---|---|---:|
| A | FedAvg | None (defenseless baseline) | ~27 min |
| B | Krum (f=2) | Distance-based single selection | ~30 min |
| C | Multi-Krum (f=2, k=7) | Distance-based top-7 selection | ~30 min |
| D | Coordinate-wise Median | Per-parameter median | ~28 min |
| E | Trimmed Mean (β=0.2) | Trim outliers, average rest | ~28 min |
| F | Bulyan (f=1) | Iterative Krum → Trimmed Mean | ~33 min |
| G | Active-Ledger PoC-Only | On-chain trust scoring, Top-7 | ~35 min |
| | **Total** | | **~3.5 hours** |

> ⚠️ **Select GPU T4 ×2 and enable Internet before running.**

**Reference (Run C, already done):** Active-Ledger PoC + Diffusion → Mean F1 = **0.89**

In [ ]:
# ============================================================
# Cell 1: GPU + Environment Verification
# ============================================================
import torch

print('=' * 60)
print('ENVIRONMENT CHECK')
print('=' * 60)
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cc   = torch.cuda.get_device_capability(0)
    print(f'GPU             : {name}')
    print(f'VRAM            : {vram:.1f} GB')
    print(f'Compute Cap     : {cc[0]}.{cc[1]}')
    if cc >= (7, 0):
        print('✅ GPU supported — everything runs on CUDA')
    else:
        print('⚠️  Older GPU — may fall back to CPU for some ops')
else:
    print('⚠️  No GPU — training will be on CPU (slower but works)')

print('=' * 60)

In [ ]:
# ============================================================
# Cell 2: Install Dependencies
# NOTE: wfdb, flwr, web3 are needed for data loading and
#       method G (PoC-Only uses Ganache + smart contract).
#       Diffusers are NOT needed for Session 1 (no diffusion).
# ============================================================
print('Installing dependencies...')

# Core FL and data dependencies
!pip install -q wfdb flwr web3 py-solc-x pyyaml scikit-learn

# Solidity compiler for smart contract deployment (method G)
from solcx import install_solc
install_solc('0.8.19')
print('✅ Solidity compiler 0.8.19 installed')

# Ganache (local Ethereum blockchain — needed only for method G)
!npm install -g ganache 2>/dev/null && echo '✅ Ganache installed' || echo '⚠️  Ganache install issue — method G may fail'

print('\n✅ All dependencies ready!')

In [ ]:
# ============================================================
# Cell 3: Copy Repository Code to Working Directory
# Upload your repo as a Kaggle Dataset before running.
# The script handles nested zip extraction automatically.
# ============================================================
import shutil, os, glob

DATASET_PATHS = glob.glob('/kaggle/input/*')
print(f'Available datasets: {DATASET_PATHS}')

if not DATASET_PATHS:
    raise FileNotFoundError(
        'No dataset found!\n'
        'Click "Add Data" → upload your zipped FYP-Blockchain-FL folder as a dataset.'
    )

# Auto-detect repo root (handles Kaggle nesting)
repo_root = None
for base in DATASET_PATHS:
    # Check base directly
    if os.path.isfile(os.path.join(base, 'main.py')):
        repo_root = base
        break
    # Check one level deep
    for item in os.listdir(base):
        candidate = os.path.join(base, item)
        if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, 'main.py')):
            repo_root = candidate
            break
    if repo_root:
        break

if repo_root is None:
    # Deep search (handles deeper nesting)
    for base in DATASET_PATHS:
        for root, dirs, files in os.walk(base):
            if root.count(os.sep) - base.count(os.sep) > 4:
                break
            if 'main.py' in files and 'core' in dirs:
                repo_root = root
                break
        if repo_root:
            break

if repo_root is None:
    raise FileNotFoundError(
        f'Cannot locate main.py in your dataset.\n'
        f'Dataset contents: {[os.listdir(p) for p in DATASET_PATHS]}'
    )

print(f'Repo root found: {repo_root}')

# Copy all files to /kaggle/working
WORK = '/kaggle/working'
copied = []
for item in os.listdir(repo_root):
    if item.startswith('.'): continue
    src = os.path.join(repo_root, item)
    dst = os.path.join(WORK, item)
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)
    copied.append(item)

os.chdir(WORK)

# Verify critical files
critical = ['main.py', 'config.yaml', 'core', 'benchmarks', 'contracts']
missing  = [f for f in critical if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f'Missing after copy: {missing}. Check your dataset.')

# Verify the new Phase 3.2 files are present
phase32_files = [
    'core/robust_aggregation.py',
    'benchmarks/run_robust_baselines.py',
]
missing_32 = [f for f in phase32_files if not os.path.exists(f)]
if missing_32:
    raise FileNotFoundError(
        f'Phase 3.2 files missing: {missing_32}\n'
        f'Make sure you uploaded the latest code (with robust_aggregation.py).'
    )

print(f'\n✅ Code copied. All critical files present.')
print(f'   Working dir: {os.getcwd()}')
print(f'   core/robust_aggregation.py : ✅')
print(f'   benchmarks/run_robust_baselines.py : ✅')

In [ ]:
# ============================================================
# Cell 4: Download + Preprocess + Partition Data
# Downloads MIT-BIH Arrhythmia Database (~120 MB),
# segments into 360-sample ECG windows,
# and partitions across 10 clients (non-IID Dirichlet).
# ============================================================
print('=' * 60)
print('STEP 1/3: Downloading MIT-BIH Arrhythmia records...')
print('=' * 60)
!rm -rf data/* checkpoints/*
!PYTHONPATH=/kaggle/working python core/download_data.py

print('\n' + '=' * 60)
print('STEP 2/3: Preprocessing ECG signals...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/preprocess_data.py

print('\n' + '=' * 60)
print('STEP 3/3: Partitioning across 10 clients (non-IID)...')
print('=' * 60)
!PYTHONPATH=/kaggle/working python core/partition_data.py

# Verify
import os
pdir = 'data/partitioned'
if os.path.exists(pdir):
    n_files = len(os.listdir(pdir))
    print(f'\n✅ Data ready — {n_files} client partition directories found')
else:
    print('❌ data/partitioned not found — check errors above')

In [ ]:
# ============================================================
# Cell 5: Start Ganache (needed ONLY for Method G: PoC-Only)
# Methods A–F do not use blockchain at all.
# The benchmark script starts Ganache mid-run when it reaches G.
# We start it here so it is ready and warm.
# ============================================================
import subprocess, time

print('Starting Ganache for Method G (Active-Ledger PoC-Only)...')
ganache_proc = subprocess.Popen(
    'NODE_OPTIONS=--max-old-space-size=8192 ganache -p 8545 '
    '--accounts 15 --deterministic --gasLimit 30000000',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(10)

if ganache_proc.poll() is not None:
    print('Retrying with ganache-cli...')
    ganache_proc = subprocess.Popen(
        'NODE_OPTIONS=--max-old-space-size=8192 ganache-cli -p 8545 '
        '--accounts 15 --deterministic --gasLimit 30000000',
        shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(10)

if ganache_proc.poll() is None:
    print(f'✅ Ganache running (PID: {ganache_proc.pid})')
else:
    print('⚠️  Ganache failed to start. Method G will run without blockchain.')
    print('   Methods A–F are unaffected.')

# Deploy FLLogger smart contract
print('\nDeploying FLLogger smart contract...')
!PYTHONPATH=/kaggle/working python core/deploy_contract.py
print('\n✅ Blockchain infrastructure ready!')

---

## 🚀 Run Session 1 — All 7 Methods

The next cell runs everything. Expected output for each method:
```
======================================================================
METHOD: A_FedAvg
======================================================================
  --- Round   1/40 [A_FedAvg] ---
  Mean F1: 0.1711  |  Latency: 38.2s  |  Per-class: [0.855 0.000 0.000 0.000 0.000]
  ...
  [A_FedAvg] Finished. Total time: 0.45h
  [A_FedAvg] Final Mean F1: 0.1711

======================================================================
METHOD: B_Krum
...
```

Results are **saved after every method** — if Kaggle times out, the `.npy` file has all completed methods.

In [ ]:
# ============================================================
# Cell 6: RUN SESSION 1 — All 7 Methods (~3.5 hours)
# This is the main experiment cell.
# DO NOT interrupt unless absolutely necessary.
# Results are checkpointed after every method.
# ============================================================
!PYTHONPATH=/kaggle/working python main.py --mode robust-baselines

In [ ]:
# ============================================================
# Cell 7: Display Results Summary
# ============================================================
import numpy as np
import os

class_names = ['Normal', 'LBBB', 'RBBB', 'APB', 'PVC']

results_path = 'checkpoints/robust_comparison.npy'
if not os.path.exists(results_path):
    print('❌ Results file not found. Did Cell 6 complete?')
else:
    results = np.load(results_path, allow_pickle=True).item()

    print('=' * 75)
    print('SESSION 1 RESULTS — Final Per-Class F1 Scores (Round 40)')
    print('=' * 75)
    print(f"{'Method':<20} {'Normal':>7} {'LBBB':>7} {'RBBB':>7} "
          f"{'APB':>7} {'PVC':>7} {'Mean F1':>9}")
    print('-' * 75)

    best_classical_name = None
    best_classical_f1   = -1

    for name, res in results.items():
        f1      = res['final_f1']
        mean_f1 = float(np.mean(f1))
        flag    = ' ← BEST classical' if name not in ['A_FedAvg', 'G_PoC_Only'] \
                  and mean_f1 > best_classical_f1 else ''

        if name not in ['A_FedAvg', 'G_PoC_Only'] and mean_f1 > best_classical_f1:
            best_classical_f1   = mean_f1
            best_classical_name = name

        print(f"{name:<20} " +
              ' '.join(f'{v:7.3f}' for v in f1) +
              f" {mean_f1:9.4f}{flag}")

    print('-' * 75)
    print(f"{'(Run C: PoC+Diffusion)':<20} "
          "  0.970   0.968   0.915   0.654   0.933     0.8887  [Phase 3.1]")
    print('=' * 75)

    if best_classical_name:
        print(f'\n🏆 Best classical baseline: {best_classical_name} '
              f'(Mean F1 = {best_classical_f1:.4f})')
        print(f'→ Report this to proceed with Session 2: '
              f'{best_classical_name} + Diffusion')
    
    print('\n📁 Output files saved to /kaggle/working/:')
    for f in ['session1_robust_comparison.pdf', 'session1_robust_comparison.png',
              'session1_latex_table.tex', 'checkpoints/robust_comparison.npy']:
        exists = '✅' if os.path.exists(f) else '❌'
        print(f'   {exists} {f}')

In [ ]:
# ============================================================
# Cell 8: Display Comparison Plot (inline)
# ============================================================
from IPython.display import Image, display
import os

if os.path.exists('session1_robust_comparison.png'):
    display(Image('session1_robust_comparison.png'))
else:
    print('Plot not generated yet. Run Cell 6 first.')

In [ ]:
# ============================================================
# Cell 9: Print LaTeX Table for Copy-Paste into Paper
# ============================================================
import os

tex_path = 'session1_latex_table.tex'
if os.path.exists(tex_path):
    with open(tex_path) as f:
        print(f.read())
else:
    print('LaTeX table not generated yet. Run Cell 6 first.')